In [1]:
import scanpy as sc
import numpy as np 
import matplotlib.pyplot as plt
import pandas as pd
import scvelo as scv
import anndata as ad
import pickle as pkl
import seaborn as sns

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

from sklearn.decomposition import PCA
import umap
from itertools import product

python_data_dir = r"C:\Users\Nikla\Documents\MoBi_Master_FS.3\Internship AG Höfer\DynaVelo_testing\data"

In [2]:
velo_gene_df = pd.read_csv(rf"{python_data_dir}\pmtr_peak_sub\chrom_velo_gene_subset.csv", index_col=0, header=0)
velo_genes = velo_gene_df.index.tolist()

In [3]:
adata_atac_all_lin = sc.read_h5ad(rf"{python_data_dir}\pmtr_peak_sub\atac.h5ad")
adata_rna_all_lin = sc.read_h5ad(rf"{python_data_dir}\pmtr_peak_sub\rna.h5ad")

adata_atac_all_lin.layers["counts"] = adata_atac_all_lin.X.copy()
sc.pp.normalize_total(adata_atac_all_lin, target_sum=np.median(adata_atac_all_lin.X.sum(1)))
sc.pp.log1p(adata_atac_all_lin)
sc.pp.pca(adata_rna_all_lin)

myeloid_cells = adata_rna_all_lin.obs_names.values[adata_rna_all_lin.obsm["X_pca"][:,1]<=1]  # filter to only myeloid lineage
adata_atac = adata_atac_all_lin[myeloid_cells,:]
adata_rna = adata_rna_all_lin[myeloid_cells,:]

atac_df = pd.DataFrame(data=adata_atac.X, index=adata_atac.obs_names.values, columns=adata_atac.var_names.values)
rna_df = pd.DataFrame(data=adata_rna.X.toarray(), index=adata_rna.obs_names.values, columns=adata_rna.var_names.values)

In [4]:
atac_velo = atac_df.loc[:, velo_genes]
rna_velo = rna_df.loc[:, velo_genes]

In [5]:
def minmax_cols(df):
    mn, mx = df.min(axis=0), df.max(axis=0)
    rng = (mx - mn).replace(0, np.nan)   # constant columns would divide by zero
    return ((df - mn) / rng).fillna(0)

atac_velo = minmax_cols(atac_velo)
rna_velo = minmax_cols(rna_velo)

In [12]:
rna_velo

,Trpc6,Pdgfd,Myct1,Rab44,Nbea,Psd3,Med12l,Pbx1,Hells,Adam22,...,Camk2d,Lncpint,Fhit,B4galnt1,Palm,Chdh,Lman2,Ly6e,Tcf7l2,Hdc
AAACAGCCACTAAGTT,0.000000,0.0,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.0,...,0.631038,0.372886,0.495332,0.000000,0.000000,0.0,0.356825,0.384890,0.000000,0.000000
AAACAGCCAGCATGAG,0.000000,0.0,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.354480,0.000000,0.397292,0.000000
AAACAGCCAGGCCATT,0.000000,0.0,0.000000,0.000000,0.00000,0.000000,0.000000,0.000000,0.000000,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000
AAACAGCCATTTAAGC,0.000000,0.0,0.000000,0.300371,0.00000,0.345214,0.000000,0.259895,0.000000,0.0,...,0.000000,0.000000,0.000000,0.000000,0.474036,0.0,0.000000,0.240466,0.000000,0.000000
AAACATGCAAGGTCCT,0.000000,0.0,0.000000,0.424607,0.00000,0.000000,0.000000,0.546493,0.000000,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTTGTGTTCTGTGCAG,0.431920,0.0,0.000000,0.000000,0.00000,0.489276,0.000000,0.000000,0.000000,0.0,...,0.000000,0.322783,0.000000,0.000000,0.000000,0.0,0.000000,0.211785,0.000000,0.000000
TTTGTTGGTAGTTGGC,0.000000,0.0,0.286807,0.000000,0.28167,0.326099,0.000000,0.245504,0.000000,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.331291,0.361600,0.000000,0.000000
TTTGTTGGTCATCATC,0.000000,0.0,0.000000,0.434118,0.00000,0.000000,0.000000,0.135641,0.207684,0.0,...,0.000000,0.000000,0.278919,0.434759,0.247402,0.0,0.420684,0.397799,0.000000,0.000000
TTTGTTGGTTGAGCCG,0.766436,0.0,0.000000,0.000000,0.00000,0.000000,0.476796,0.358060,0.000000,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.495052,0.000000,0.000000


## kNN 50 Smoothing of Signals
To calculate chromatin potential, we first smoothed DORC scores (chromatin space) and corresponding gene expression (RNA space) over a k-nearest neighbor graph (k-NN, k=50), calculated using normalized ATAC topics from cisTopic.

## kNN 10 Assignment of ATAC to RNA
Find the 10 RNA profiles with the highest correlation to the current ATAC cell and compute their average as the predicted future state of the ATAC. The difference between the two points results in a velocity / chromatin potential.

In [20]:
A = atac_velo.values   # (n_cells, n_genes)
R = rna_velo.values     # (n_cells, n_genes)

A_std = (A - A.mean(axis=1, keepdims=True)) / A.std(axis=1, keepdims=True)
R_std = (R - R.mean(axis=1, keepdims=True)) / R.std(axis=1, keepdims=True)

corrs_axr = pd.DataFrame(
    A_std @ R_std.T / A.shape[1],
    index=atac_velo.index,  # atac cells
    columns=rna_velo.index,  # rna cells 
)

In [26]:
nns = {}

for cell in corrs_axr.index: 
    pot_partners = corrs_axr.loc[cell, :]
    nn10s = pot_partners.sort_values(ascending=False).index[:10]
    nns[cell] = nn10s